In [32]:
"""
Mtb Q-Loop Pharmacophore Pipeline v7.0
Fixed from v6.1:
  1. SMARTS pattern overlap (Donor vs Hydrophobe matching same atoms) → fixed patterns
  2. Feature diversity enforcement (never collapse to single feature family)
  3. MIN_FEATURE_DISTANCE reduced; replaced with family-aware deduplication
  4. NaN Spearman correlation (all-identical binary scores) → added continuous scoring
  5. Aromatic features not appearing → fixed SMARTS + verified pattern parsing
  6. Feature factory silent parse failures → validation step added
  7. Improved coverage/importance threshold logic
  8. Cross-validation NaN propagation → guarded before mean computation
"""

import os
import warnings
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, rdFMCS, ChemicalFeatures
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split, KFold
from collections import defaultdict
from scipy.stats import spearmanr

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CONFORMERS_DIR = "Conformers"
TEMPLATE_NAME  = "CK_2_63.sdf"
EXCEL_PATH     = "Mtb.xlsx"

RANDOM_SEED    = 42
TRAIN_SIZE     = 0.7

CLUSTERING_EPS          = 1.6   # Å – DBSCAN neighbourhood radius
MIN_RADIUS              = 1.0   # Å – minimum pharmacophore sphere radius
MAX_RADIUS              = 2.5   # Å – maximum pharmacophore sphere radius
MIN_FEATURE_COVERAGE    = 0.5   # fraction of training molecules that must hit a feature
MIN_FEATURE_COVERAGE_CV = 0.4   # slightly relaxed for cross-validation folds

# FIX 3 – replaced global MIN_FEATURE_DISTANCE with family-aware dedup (see below)
# Within the same feature family we still deduplicate spatially, but NEVER remove a
# feature simply because it is close to a *different* family.
INTRA_FAMILY_MIN_DIST   = 2.0   # Å – min distance between two features of the SAME family

ALIGNMENT_RMSD_CUTOFF   = 5.0   # Å – discard alignments worse than this

# ─────────────────────────────────────────────────────────────────────────────
# FIX 1 – CORRECTED FEATURE DEFINITIONS
#
# Root cause of v6.1 bug: the Hydrophobe pattern matched sp3 carbons that are
# directly bonded to N-H donor atoms, so the two feature centres ended up <3 Å
# apart and the distance filter wiped the Donor every time.
#
# Changes:
#   • Donor: kept standard N-H / O-H / S-H patterns; added explicit H-count guards
#   • Acceptor: tightened to heteroatoms with lone pairs only
#   • Hydrophobe: EXCLUDED atoms bonded to any heteroatom (was matching α-carbons
#     of donor/acceptor groups); now truly aliphatic / halogen / aromatic-C only
#   • Aromatic: single [a] covers all aromatic atoms (was silently not parsed in v6.1
#     because DefineFeature requires exactly one SMARTS per line in some RDKit builds –
#     fixed by using one SMARTS token per DefineFeature block)
#   • Cationic / Anionic: kept, minor tightening
#
# FIX 5 – each DefineFeature block now has ONE SMARTS token (the multi-SMARTS
# syntax "$([A])  $([B])" is not reliably supported across all RDKit versions;
# use OR syntax "[#7;...],$([#8;...])" within a single SMARTS instead).
# ─────────────────────────────────────────────────────────────────────────────

IMPROVED_FDEF = """
AtomType NDonor [$([N;!H0;v3,v4;+0,+1])]
AtomType ODonor [$([O,S;H1;+0])]
DefineFeature Donor [$([N;!H0;v3,v4;+0,+1]),$(O[H]),$(S[H])]
  Family Donor
  Weights 1.0
EndFeature

DefineFeature Acceptor [$([N;H0;+0;v3]),$([O;H0;+0;v2]),$([F;$(F-[#6]);!$(FC[F,Cl,Br,I])]),$([S;H0;+0;v2])]
  Family Acceptor
  Weights 1.0
EndFeature

DefineFeature Aromatic [a]
  Family Aromatic
  Weights 1.0
EndFeature

DefineFeature Hydrophobe [$([c,s,br,I]),$([C;D3,D4;!$([C,N,O]=[C,N,O,S]);!$([CH2][O,N,S]);!$([CH][O,N,S]);!$([C][O,N,S])]),$([F,Cl,Br,I;$([F,Cl,Br,I]-[#6;!$([#6]~[#7,#8,#16])])])]
  Family Hydrophobe
  Weights 0.8
EndFeature

DefineFeature Cationic [$([NH2;+1]),$([NH3;+1]),$([NH4;+1]),$([nH;+1])]
  Family Cationic
  Weights 1.2
EndFeature

DefineFeature Anionic [$([C](=O)[O-]),$([P](=O)[O-]),$([S](=O)[O-]),$([c](=O)[O-])]
  Family Anionic
  Weights 1.2
EndFeature
"""

# ─────────────────────────────────────────────────────────────────────────────
# FIX 5 – validate feature factory at startup so silent parse failures surface
# ─────────────────────────────────────────────────────────────────────────────

def build_and_validate_factory(fdef_string=IMPROVED_FDEF):
    """Build ChemicalFeatures factory and do a quick smoke-test."""
    factory = ChemicalFeatures.BuildFeatureFactoryFromString(fdef_string)
    # Smoke-test on a simple molecule that should hit all 6 families
    # benzylamine: Donor (NH2), Acceptor (none here, use pyridine), Aromatic, Hydrophobe
    test_smiles = {
        'Donor':      'Nc1ccccc1',       # aniline – N-H donor
        'Acceptor':   'c1ccncc1',        # pyridine – N acceptor
        'Aromatic':   'c1ccccc1',        # benzene
        'Hydrophobe': 'CCCC',            # butane – aliphatic C
        'Cationic':   '[NH3+]CCC',       # protonated amine
        'Anionic':    'CC(=O)[O-]',      # acetate
    }
    missing = []
    for family, smi in test_smiles.items():
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            missing.append(family)
            continue
        feats = factory.GetFeaturesForMol(mol)
        families_found = {f.GetFamily() for f in feats}
        if family not in families_found:
            missing.append(family)
    if missing:
        print(f"  ⚠ Feature factory smoke-test: families NOT detected on test molecules: {missing}")
        print("    → Check SMARTS patterns for these families.")
    else:
        print("  ✓ Feature factory smoke-test passed (all 6 families detected)")
    return factory


# ─────────────────────────────────────────────────────────────────────────────
# CORE UTILS
# ─────────────────────────────────────────────────────────────────────────────

def find_file_fuzzy(molecule_name, directory):
    target = str(molecule_name).lower().strip().replace('.sdf', '')
    variants = {target, target.replace('-', '_'), target.replace('_', '-')}
    for filename in os.listdir(directory):
        f_lower = filename.lower()
        if f_lower.endswith('.sdf'):
            name_part = f_lower.replace('.sdf', '')
            if name_part in variants or target in name_part:
                return os.path.join(directory, filename)
    return None


def robust_load_ensemble(path, name=None):
    suppl = Chem.SDMolSupplier(path, removeHs=False, sanitize=False)
    mols = [m for m in suppl if m is not None]
    if not mols:
        return None
    base = mols[0]
    try:
        Chem.SanitizeMol(base)
    except Exception:
        try:
            Chem.SanitizeMol(
                base,
                Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE
            )
        except Exception:
            return None
    for i in range(1, len(mols)):
        try:
            base.AddConformer(mols[i].GetConformer(), assignId=True)
        except Exception:
            continue
    if name:
        base.SetProp("_Name", name)
    return base


def align_ensemble(probe_mol, template_mol):
    """Align every conformer of probe_mol to template; return (mol, best_rmsd)."""
    mcs = rdFMCS.FindMCS([template_mol, probe_mol], timeout=2)
    if mcs.numAtoms < 3:
        return probe_mol, 999.0
    patt = Chem.MolFromSmarts(mcs.smartsString)
    if patt is None:
        return probe_mol, 999.0
    probe_match    = probe_mol.GetSubstructMatch(patt)
    template_match = template_mol.GetSubstructMatch(patt)
    if not probe_match or not template_match:
        return probe_mol, 999.0
    atom_map = list(zip(probe_match, template_match))
    best_rmsd = 999.0
    for conf in probe_mol.GetConformers():
        try:
            rmsd = rdMolAlign.AlignMol(
                probe_mol, template_mol,
                prbCid=conf.GetId(), refCid=0,
                atomMap=atom_map
            )
            if rmsd < best_rmsd:
                best_rmsd = rmsd
        except Exception:
            continue
    return probe_mol, best_rmsd


def load_and_align(name, template):
    """Load compound, align to template; return (mol, rmsd) or (None, None)."""
    path = find_file_fuzzy(name, CONFORMERS_DIR)
    if not path:
        return None, None
    mol = robust_load_ensemble(path, name)
    if not mol:
        return None, None
    mol, rmsd = align_ensemble(mol, template)
    if rmsd > ALIGNMENT_RMSD_CUTOFF:
        return None, None
    return mol, rmsd


# ─────────────────────────────────────────────────────────────────────────────
# FIX 2 & 3 – FAMILY-AWARE FEATURE CONSTRAINT ENFORCEMENT
#
# Old logic: drop any feature whose centre is within MIN_FEATURE_DISTANCE of ANY
# already-kept feature, regardless of feature type. This systematically removed
# Donor whenever a Hydrophobe was nearby (even though they are chemically distinct).
#
# New logic:
#   • Within the SAME family: apply spatial deduplication (keep higher-importance one).
#   • Across DIFFERENT families: NEVER remove a feature purely for proximity.
#   • After intra-family dedup: optionally warn (not remove) about cross-family proximity
#     so the user is aware but the model retains chemical diversity.
# ─────────────────────────────────────────────────────────────────────────────

def enforce_feature_constraints(points, min_coverage=MIN_FEATURE_COVERAGE):
    """
    Deduplicate pharmacophore points:
      1. Drop low-coverage candidates (< min_coverage).
      2. Within the same feature family, merge/drop spatially redundant points
         (keeping the higher-importance one within INTRA_FAMILY_MIN_DIST).
      3. Cross-family proximity: warn only, never remove.
    """
    # Step 1: coverage filter
    coverage_passed = [p for p in points if p['coverage'] >= min_coverage]
    dropped_coverage = [p for p in points if p['coverage'] < min_coverage]
    for p in dropped_coverage:
        print(f"  → Dropping {p['family']} (coverage {p['coverage']:.1%} < {min_coverage:.0%})")

    if not coverage_passed:
        return []

    # Step 2: intra-family spatial deduplication
    by_family = defaultdict(list)
    for p in coverage_passed:
        by_family[p['family']].append(p)

    deduplicated = []
    for family, pts in by_family.items():
        # sort by importance descending
        pts_sorted = sorted(pts, key=lambda x: -x['importance'])
        kept = []
        for pt in pts_sorted:
            too_close = False
            for existing in kept:
                dist = np.linalg.norm(pt['center'] - existing['center'])
                if dist < INTRA_FAMILY_MIN_DIST:
                    print(f"  → Merging {family} cluster (distance {dist:.2f}Å, keeping higher-importance centre)")
                    too_close = True
                    break
            if not too_close:
                kept.append(pt)
        deduplicated.extend(kept)

    # Step 3: cross-family proximity warning
    for i, pi in enumerate(deduplicated):
        for j, pj in enumerate(deduplicated):
            if j <= i:
                continue
            if pi['family'] != pj['family']:
                dist = np.linalg.norm(pi['center'] - pj['center'])
                if dist < 1.5:
                    print(f"  ⚠ Note: {pi['family']} and {pj['family']} centres are only {dist:.2f}Å apart "
                          f"– both retained (different chemical roles)")

    return deduplicated


# ─────────────────────────────────────────────────────────────────────────────
# FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def extract_features_with_diagnostics(aligned_mols, factory, min_coverage=MIN_FEATURE_COVERAGE):
    """Extract and cluster pharmacophore features; return model point list."""
    all_obs = defaultdict(list)
    n_mols = len(aligned_mols)

    for idx, (mol, pic50, name) in enumerate(aligned_mols):
        feats = factory.GetFeaturesForMol(mol)
        for conf in mol.GetConformers():
            cid = conf.GetId()
            for f in feats:
                all_obs[f.GetFamily()].append({
                    'pos':   np.array(f.GetPos(cid)),
                    'pIC50': pic50,
                    'idx':   idx,
                    'name':  name,
                })

    # Report which families were detected
    print(f"  Feature families detected: {list(all_obs.keys())}")

    model_points = []
    print(f"\n{'#'*20} PHARMACOPHORE MODEL SUMMARY {'#'*20}")
    print(f"{'ID':<4} {'Type':<12} {'X':>8} {'Y':>8} {'Z':>8} {'Radius':>8} {'Coverage':>10} {'Importance':>12}")
    print("-" * 80)

    feat_counter = 1
    for family, obs in all_obs.items():
        if len(obs) < 2:
            continue
        coords   = np.array([o['pos'] for o in obs])
        clusters = DBSCAN(eps=CLUSTERING_EPS, min_samples=2).fit(coords)

        for label in set(clusters.labels_):
            if label == -1:
                continue
            mask     = clusters.labels_ == label
            c_pic50  = np.array([obs[i]['pIC50'] for i in np.where(mask)[0]])
            center   = np.average(coords[mask], axis=0, weights=c_pic50)
            dists    = np.linalg.norm(coords[mask] - center, axis=1)
            radius   = min(max(np.mean(dists) * 1.5, MIN_RADIUS), MAX_RADIUS)
            u_mols   = len(set(obs[i]['idx'] for i in np.where(mask)[0]))
            coverage   = u_mols / n_mols
            importance = np.mean(c_pic50) * coverage

            # Show all candidates (even below threshold) for transparency
            flag = '' if coverage >= min_coverage else ' (below threshold)'
            print(f"{feat_counter:<4} {family:<12} {center[0]:>8.2f} {center[1]:>8.2f} {center[2]:>8.2f} "
                  f"{radius:>8.2f} {coverage:>9.1%} {importance:>12.3f}{flag}")
            feat_counter += 1

            model_points.append({
                'family':     family,
                'center':     center,
                'radius':     radius,
                'coverage':   coverage,
                'importance': importance,
            })

    print(f"\nApplying constraints (min coverage: {min_coverage:.0%}, "
          f"intra-family min dist: {INTRA_FAMILY_MIN_DIST}Å)...")
    final_points = enforce_feature_constraints(model_points, min_coverage=min_coverage)
    print(f"\nFinal model: {len(final_points)} features selected from {len(model_points)} candidates")
    print("#" * 80)
    return final_points


# ─────────────────────────────────────────────────────────────────────────────
# FIX 4 – CONTINUOUS (DISTANCE-BASED) SCORING TO AVOID NaN SPEARMAN
#
# Old binary score: 1 if any conformer feature falls within radius, else 0.
# Problem: with a single feature the score is always 0 or 1 → all-identical
# validation sets → Spearman is undefined (NaN).
#
# New score: for each model point, find the CLOSEST matching feature across
# all conformers and compute a Gaussian-decay score based on that distance.
# Sum across all model points and normalise. This gives a continuous value
# even when no conformer perfectly hits the sphere, enabling real rank correlation.
# ─────────────────────────────────────────────────────────────────────────────

def score_molecule_continuous(mol, points, factory):
    """
    Continuous pharmacophore fit score in [0, 1].

    For each model point p_i:
      - Find the minimum distance d from any same-family feature across all conformers.
      - Contribution = exp(-(d / radius_i)^2)   [= 1.0 when d=0, ~0.37 at d=radius]

    Final score = mean contribution across all model points.
    Also returns integer hit count (distance < radius).
    """
    if not points:
        return 0.0, 0

    contributions = []
    hit_count = 0

    for pt in points:
        min_dist = np.inf
        for conf in mol.GetConformers():
            cid  = conf.GetId()
            feats = factory.GetFeaturesForMol(mol)
            for f in feats:
                if f.GetFamily() == pt['family']:
                    pos  = np.array(f.GetPos(cid))
                    dist = np.linalg.norm(pos - pt['center'])
                    if dist < min_dist:
                        min_dist = dist
        if min_dist == np.inf:
            contributions.append(0.0)
        else:
            contrib = np.exp(-(min_dist / pt['radius']) ** 2)
            contributions.append(contrib)
            if min_dist <= pt['radius']:
                hit_count += 1

    return float(np.mean(contributions)), hit_count


# ─────────────────────────────────────────────────────────────────────────────
# ACTIVITY CLIFF CHECK
# ─────────────────────────────────────────────────────────────────────────────

def check_activity_cliffs(train_df, model_points, factory, template, threshold=0.25):
    """Validate discrimination between actives (pIC50 > 7) and inactives (pIC50 < 6)."""
    active_comps   = train_df[train_df['pIC50'] > 7.0]
    inactive_comps = train_df[train_df['pIC50'] < 6.0]

    if len(active_comps) < 2 or len(inactive_comps) < 2:
        print("  ⚠ Warning: Insufficient active/inactive compounds for cliff detection")
        return True

    def collect_scores(subset):
        scores = []
        for _, row in subset.iterrows():
            mol, _ = load_and_align(row['Name'], template)
            if mol:
                fit, _ = score_molecule_continuous(mol, model_points, factory)
                scores.append(fit)
        return scores

    active_scores   = collect_scores(active_comps)
    inactive_scores = collect_scores(inactive_comps)

    if active_scores and inactive_scores:
        mean_active   = np.mean(active_scores)
        mean_inactive = np.mean(inactive_scores)
        separation    = mean_active - mean_inactive
        print(f"  Activity cliff check: Active avg score = {mean_active:.3f}, "
              f"Inactive avg = {mean_inactive:.3f}")
        print(f"  Separation = {separation:.3f}")
        if separation < threshold:
            print("  ⚠ Warning: Poor discrimination between active and inactive compounds")
            return False
        else:
            print("  ✓ Good discrimination between active and inactive compounds")
            return True
    return False


# ─────────────────────────────────────────────────────────────────────────────
# FIX 7 & 8 – CROSS-VALIDATION WITH NaN GUARD
# ─────────────────────────────────────────────────────────────────────────────

def cross_validate_pharmacophore(df, factory, k=5):
    """
    K-fold cross-validation.
    Fixes vs v6.1:
      • Uses continuous score (not binary) so Spearman is always defined.
      • Guards against NaN before computing CV mean/std.
      • Relaxes coverage threshold slightly for smaller folds.
    """
    print(f"\n{'#'*20} CROSS-VALIDATION (k={k}) {'#'*20}")

    valid_rows = [row for _, row in df.iterrows() if find_file_fuzzy(row['Name'], CONFORMERS_DIR)]
    if len(valid_rows) < 10:
        print("  Insufficient compounds for cross-validation")
        return []
    valid_df = pd.DataFrame(valid_rows)

    # Load template once
    t_path = find_file_fuzzy(TEMPLATE_NAME, CONFORMERS_DIR)
    if not t_path:
        print("  Template not found – skipping CV")
        return []
    template = robust_load_ensemble(t_path)
    if not template:
        print("  Could not load template – skipping CV")
        return []

    kf = KFold(n_splits=min(k, len(valid_df)), shuffle=True, random_state=RANDOM_SEED)
    cv_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(valid_df)):
        train_fold = valid_df.iloc[train_idx]
        val_fold   = valid_df.iloc[val_idx]

        # Build aligned training set for this fold
        aligned_train = []
        for _, row in train_fold.iterrows():
            mol, _ = load_and_align(row['Name'], template)
            if mol:
                aligned_train.append((mol, row['pIC50'], row['Name']))

        if len(aligned_train) < 3:
            print(f"  Fold {fold+1}: Insufficient training data, skipping")
            continue

        # FIX 7 – relaxed coverage for smaller folds
        fold_coverage = max(MIN_FEATURE_COVERAGE_CV,
                            MIN_FEATURE_COVERAGE * len(aligned_train) / len(valid_df))
        model = extract_features_with_diagnostics(aligned_train, factory,
                                                  min_coverage=fold_coverage)
        if not model:
            print(f"  Fold {fold+1}: No model generated")
            continue

        # Score validation molecules
        val_results = []
        for _, row in val_fold.iterrows():
            mol, _ = load_and_align(row['Name'], template)
            if mol:
                fit, hits = score_molecule_continuous(mol, model, factory)
                val_results.append({'pIC50': row['pIC50'], 'fit': fit, 'hits': hits})

        if len(val_results) < 2:
            print(f"  Fold {fold+1}: Too few validation results (n={len(val_results)})")
            continue

        fits   = [r['fit']   for r in val_results]
        pic50s = [r['pIC50'] for r in val_results]

        # FIX 8 – guard against constant fit vectors before calling spearmanr
        if len(set(fits)) < 2:
            print(f"  Fold {fold+1}: All fit scores identical ({fits[0]:.3f}) – "
                  f"Spearman undefined, skipping fold")
            continue

        rho, pval = spearmanr(fits, pic50s)

        # Guard against any remaining NaN
        if np.isnan(rho):
            print(f"  Fold {fold+1}: Spearman returned NaN unexpectedly, skipping")
            continue

        cv_scores.append(rho)
        print(f"  Fold {fold+1}: Spearman ρ = {rho:.3f}  (p={pval:.4f}, n={len(val_results)})")

    if cv_scores:
        print(f"\nCV Summary: Mean ρ = {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}  "
              f"(valid folds: {len(cv_scores)}/{k})")
    else:
        print("\nCV Summary: No valid folds produced a score")
    print("#" * 60)
    return cv_scores


# ─────────────────────────────────────────────────────────────────────────────
# MAIN EXECUTION
# ─────────────────────────────────────────────────────────────────────────────

def run():
    print("=" * 80)
    print("Mtb Q-Loop Pharmacophore Pipeline v7.0")
    print("=" * 80)

    # ── Build & validate feature factory ──────────────────────────────────────
    print("\nBuilding and validating feature factory...")
    factory = build_and_validate_factory()

    # ── Load data ─────────────────────────────────────────────────────────────
    df = pd.read_excel(EXCEL_PATH, sheet_name=1)
    df = df[df['Binding site'].str.upper() == 'Q-LOOP'].copy()
    ic50_numeric = pd.to_numeric(
        df['IC50 μM'].astype(str).str.replace('μM', '', regex=False),
        errors='coerce'
    )
    df['pIC50'] = -np.log10(ic50_numeric * 1e-6)
    df = df.dropna(subset=['pIC50'])
    print(f"\nDataset: {len(df)} compounds  |  "
          f"pIC50 range: {df['pIC50'].min():.2f} – {df['pIC50'].max():.2f}")

    # ── Train / validation split ──────────────────────────────────────────────
    train_df, val_df = train_test_split(df, train_size=TRAIN_SIZE, random_state=RANDOM_SEED)
    print(f"Training set  : {len(train_df)} compounds")
    print(f"Validation set: {len(val_df)} compounds")

    # ── Load template ─────────────────────────────────────────────────────────
    t_path = find_file_fuzzy(TEMPLATE_NAME, CONFORMERS_DIR)
    if not t_path:
        print(f"Error: Template '{TEMPLATE_NAME}' not found in '{CONFORMERS_DIR}'")
        return
    template = robust_load_ensemble(t_path)
    if not template:
        print("Error: Could not load template molecule")
        return

    # ── Align training molecules ──────────────────────────────────────────────
    print("\nAligning training molecules to template...")
    aligned_train = []
    for _, row in train_df.iterrows():
        mol, rmsd = load_and_align(row['Name'], template)
        if mol:
            aligned_train.append((mol, row['pIC50'], row['Name']))
            print(f"  {row['Name']:<20} aligned  RMSD = {rmsd:.2f} Å")
        else:
            print(f"  {row['Name']:<20} FAILED (not found or poor alignment)")

    print(f"\nSuccessfully aligned {len(aligned_train)}/{len(train_df)} compounds")
    if len(aligned_train) < 3:
        print("Error: Insufficient aligned compounds for pharmacophore generation")
        return

    # ── Generate pharmacophore model ──────────────────────────────────────────
    model = extract_features_with_diagnostics(aligned_train, factory)
    if not model:
        print("Error: No pharmacophore features generated")
        return

    print(f"\nFinal pharmacophore model — {len(model)} features:")
    for i, pt in enumerate(model, 1):
        cx, cy, cz = pt['center']
        print(f"  {i}. {pt['family']:<12}  centre=({cx:.2f}, {cy:.2f}, {cz:.2f})  "
              f"radius={pt['radius']:.2f} Å  coverage={pt['coverage']:.1%}")

    # ── Activity cliff validation ─────────────────────────────────────────────
    print("\n" + "#"*20 + " ACTIVITY VALIDATION " + "#"*20)
    good_discrimination = check_activity_cliffs(train_df, model, factory, template)

    # ── Hold-out validation ───────────────────────────────────────────────────
    print("\n" + "#"*20 + " VALIDATION RESULTS " + "#"*20)
    results = []
    for _, row in val_df.iterrows():
        mol, _ = load_and_align(row['Name'], template)
        if mol:
            fit, hits = score_molecule_continuous(mol, model, factory)
            results.append({
                'name':  row['Name'],
                'ic50':  10 ** (-row['pIC50'] + 6),
                'pIC50': row['pIC50'],
                'fit':   fit,
                'hits':  hits,
            })

    if results:
        print(f"\n{'Name':<20} {'IC50 (µM)':>11} {'pIC50':>7} {'Fit Score':>10} {'Matches':>10}")
        print("-" * 65)
        for r in sorted(results, key=lambda x: x['ic50']):
            match_str = f"{r['hits']}/{len(model)}"
            print(f"{r['name']:<20} {r['ic50']:>11.4f} {r['pIC50']:>7.2f} {r['fit']:>10.3f} {match_str:>10}")

        fits   = [r['fit']   for r in results]
        pic50s = [r['pIC50'] for r in results]

        if len(set(fits)) >= 2:
            rho, pval = spearmanr(fits, pic50s)
            print(f"\nSpearman rank correlation: ρ = {rho:.3f}  (p = {pval:.4f})")
        else:
            print("\nAll fit scores identical – Spearman undefined (model not discriminating)")

        active_cutoff = 1.0
        active   = [r for r in results if r['ic50'] <  active_cutoff]
        inactive = [r for r in results if r['ic50'] >= active_cutoff]

        if active and inactive:
            af = [r['fit'] for r in active]
            inf= [r['fit'] for r in inactive]
            print(f"\nActive   (< {active_cutoff} µM): mean fit = {np.mean(af):.3f} ± {np.std(af):.3f}")
            print(f"Inactive (≥ {active_cutoff} µM): mean fit = {np.mean(inf):.3f} ± {np.std(inf):.3f}")

            top_pct = 30
            n_top = max(1, int(len(results) * top_pct / 100))
            top_compounds  = sorted(results, key=lambda x: -x['fit'])[:n_top]
            n_active_top   = sum(1 for c in top_compounds if c['ic50'] < active_cutoff)
            if active:
                enrichment = n_active_top / (len(active) * n_top / len(results))
                print(f"Enrichment factor (top {top_pct}%): {enrichment:.2f}")
    else:
        print("No validation results generated")

    # ── Cross-validation ──────────────────────────────────────────────────────
    cv_scores = cross_validate_pharmacophore(df, factory)

    # ── Final assessment ──────────────────────────────────────────────────────
    print("\n" + "#"*20 + " FINAL ASSESSMENT " + "#"*20)

    if cv_scores:
        mean_cv = np.mean(cv_scores)
        if mean_cv > 0.5:
            print(f"✓ Model shows good predictive power  (CV ρ = {mean_cv:.3f})")
        elif mean_cv > 0.3:
            print(f"⚠ Model shows moderate predictive power  (CV ρ = {mean_cv:.3f}) "
                  f"– consider feature optimisation")
        else:
            print(f"✗ Model shows poor predictive power  (CV ρ = {mean_cv:.3f}) "
                  f"– revise features or expand dataset")
    else:
        print("⚠ No valid CV folds – cannot assess predictive power from cross-validation")

    if not good_discrimination:
        print("⚠ Warning: Model may overfit – consider reducing feature complexity")

    # ── Feature diversity summary ─────────────────────────────────────────────
    families_in_model = [p['family'] for p in model]
    print(f"\nFeature diversity: {len(set(families_in_model))} distinct families "
          f"in final model: {sorted(set(families_in_model))}")
    if len(set(families_in_model)) < 2:
        print("  ⚠ Only one feature family in model – consider lowering "
              "MIN_FEATURE_COVERAGE or expanding dataset for richer pharmacophore")

    print("\nPipeline complete!")


if __name__ == "__main__":
    run()

Mtb Q-Loop Pharmacophore Pipeline v7.0

Building and validating feature factory...
  ⚠ Feature factory smoke-test: families NOT detected on test molecules: ['Acceptor', 'Hydrophobe']
    → Check SMARTS patterns for these families.

Dataset: 22 compounds  |  pIC50 range: 4.80 – 8.52
Training set  : 15 compounds
Validation set: 7 compounds

Aligning training molecules to template...
  RKA-70               aligned  RMSD = 0.17 Å
  RKA-259              FAILED (not found or poor alignment)
  RKA-307              aligned  RMSD = 0.00 Å
  WDH-1U-10            aligned  RMSD = 0.02 Å
  RKA-73               aligned  RMSD = 0.11 Å
  WDH-1W-5             FAILED (not found or poor alignment)
  SL-2-25              aligned  RMSD = 2.98 Å
  CK-3-14              aligned  RMSD = 0.03 Å
  CK-2-63              aligned  RMSD = 0.00 Å
  WDH-2R-4             aligned  RMSD = 2.48 Å
  CK-2-88              aligned  RMSD = 0.00 Å
  PG-203               aligned  RMSD = 1.03 Å
  GN-171               aligned  RMSD